In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")
from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import get_telecommute

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

In [ ]:
data_fullsurvey['Person'] = data_fullsurvey['Person'].merge(data_fullsurvey['Household'][['hhno', 'hhparcel', 'hhtaz']], on='hhno', how='left')
data_fullsurvey['Person'] = data_fullsurvey['Person'].merge(taz_subarea[['TAZ', 'County', 'Jurisdiction', 'DistrictFlowName', 'Subarea']], 
                                                    left_on='hhtaz', right_on='TAZ', how='left')

In [ ]:
data_daysim['Person'] = data_daysim['Person'].merge(data_daysim['Household'][['hhno', 'hhparcel', 'hhtaz']], on='hhno', how='left')
data_daysim['Person'] = data_daysim['Person'].merge(taz_subarea[['TAZ', 'County', 'Jurisdiction', 'DistrictFlowName', 'Subarea']], 
                                                    left_on='hhtaz', right_on='TAZ', how='left')
data_daysim['PersonDay'] = data_daysim['PersonDay'].merge(data_daysim['Person'][['hhno', 'pno', 'pwtyp', 'pwpcl', 'hhparcel', 'hhtaz', 
                                                            'County', 'Jurisdiction', 'Subarea']], 
                                                            on=['hhno', 'pno'], how='left')

In [ ]:
def show_wfh_table(df_survey, df_daysim):
    workers_survey = df_survey[['worker_type', 'psexpfac']].groupby('worker_type').sum().reset_index()
    workers_daysim = df_daysim[['worker_type', 'pdexpfac']].groupby('worker_type').sum().reset_index()
    workers_survey.columns = ['Telecommute Type', 'Number of Workers']
    workers_daysim.columns = ['Telecommute Type', 'Number of Workers']
    workers = workers_survey.merge(workers_daysim, on='Telecommute Type', how='left', suffixes=(' (Survey)', ' (Model)'))
    # show numbers
    workers = workers.set_index('Telecommute Type')
    workers = workers.loc[['Commuter', 'Telecommuter', 'WFH', 'Not Worker'], :]
    display(workers.style.format({'Number of Workers (Survey)': '{:,.0f}',
                                  'Number of Workers (Model)': '{:,.0f}'}))

def show_worker_table(df_survey, df_daysim):
    workers_survey = df_survey[['pwtyp', 'psexpfac']].groupby('pwtyp').sum().reset_index()
    workers_daysim = df_daysim[['pwtyp', 'psexpfac']].groupby('pwtyp').sum().reset_index()
    workers_survey.columns = ['Worker Type', 'Number of Workers']
    workers_daysim.columns = ['Worker Type', 'Number of Workers']
    workers = workers_survey.merge(workers_daysim, on='Worker Type', how='left', suffixes=(' (Survey)', ' (Model)'))
    # show numbers
    workers = workers.set_index('Worker Type')
    workers = workers.loc[['Paid Full-Time Worker', 'Paid Part-Time Worker', 'Not a Paid Worker'], :]
    display(workers.style.format({'Number of Workers (Survey)': '{:,.0f}',
                                  'Number of Workers (Model)': '{:,.0f}'}))

In [ ]:
data_daysim['PersonDay'] = get_telecommute(data_daysim['PersonDay'])
data_fullsurvey['Person'] = get_telecommute(data_fullsurvey['Person'])

## PSRC Region

In [ ]:
show_worker_table(data_fullsurvey['Person'], data_daysim['Person'])

In [ ]:
show_wfh_table(df_survey=data_fullsurvey['Person'], df_daysim=data_daysim['PersonDay'])

## King County

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']=='King']
df_daysim = data_daysim['Person'][data_daysim['Person']['County']=='King']
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']=='King']
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['County']=='King']
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## Out of King County

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']!='King']
df_daysim = data_daysim['Person'][data_daysim['Person']['County']!='King']
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']!='King']
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['County']!='King']
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## BKR Area

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## Out of BKR Area

In [ ]:
df_survey = data_fullsurvey['Person'][~data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['Person'][~data_daysim['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][~data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['PersonDay'][~data_daysim['PersonDay']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## City of Bellevue

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['BELLEVUE'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['BELLEVUE'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## City of Kirkland

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['KIRKLAND'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['KIRKLAND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['KIRKLAND'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['KIRKLAND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## City of Redmond

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['KIRKLAND'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['KIRKLAND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['REDMOND'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['REDMOND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

## Bellevue Downtown

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Subarea']==3]
df_daysim = data_daysim['Person'][data_daysim['Person']['Subarea']==3]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

In [ ]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Subarea']==3]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Subarea']==3]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)